# PyTorch: Training Loop

**Interview focus:** DataLoader, optimizer, training/eval modes, gradient clipping, and a complete training loop.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

## 1. Synthetic Dataset & DataLoader

In [ ]:
torch.manual_seed(42)

# Binary classification: 2D points, label = 1 if x1 + x2 > 0
n = 1000
X = torch.randn(n, 2)
y = (X[:, 0] + X[:, 1] > 0).long()

dataset = TensorDataset(X, y)
loader = DataLoader(dataset, batch_size=32, shuffle=True)

for batch_x, batch_y in loader:
  print(f"batch_x: {batch_x.shape}, batch_y: {batch_y.shape}")
  break

## 2. Model, Loss, Optimizer

In [ ]:
class Classifier(nn.Module):
  def __init__(self):
    super().__init__()
    self.net = nn.Sequential(
      nn.Linear(2, 32),
      nn.ReLU(),
      nn.Linear(32, 32),
      nn.ReLU(),
      nn.Linear(32, 2),
    )

  def forward(self, x):
    return self.net(x)

model = Classifier()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

## 3. Training Loop

In [ ]:
def train_epoch(model, loader, criterion, optimizer):
  model.train()
  total_loss, correct, total = 0.0, 0, 0

  for batch_x, batch_y in loader:
    optimizer.zero_grad()
    logits = model(batch_x)
    loss = criterion(logits, batch_y)
    loss.backward()
    optimizer.step()

    total_loss += loss.item() * batch_x.size(0)
    correct += (logits.argmax(dim=1) == batch_y).sum().item()
    total += batch_x.size(0)

  return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
  model.eval()
  total_loss, correct, total = 0.0, 0, 0

  for batch_x, batch_y in loader:
    logits = model(batch_x)
    loss = criterion(logits, batch_y)

    total_loss += loss.item() * batch_x.size(0)
    correct += (logits.argmax(dim=1) == batch_y).sum().item()
    total += batch_x.size(0)

  return total_loss / total, correct / total

In [ ]:
epochs = 20
for epoch in range(1, epochs + 1):
  train_loss, train_acc = train_epoch(model, loader, criterion, optimizer)
  val_loss, val_acc = evaluate(model, loader, criterion)
  if epoch % 5 == 0 or epoch == 1:
    print(f"Epoch {epoch:2d} | train loss: {train_loss:.4f} acc: {train_acc:.3f} | val acc: {val_acc:.3f}")

## 4. Optimizer Details

In [ ]:
# SGD with momentum
sgd = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=1e-4)

# Learning rate scheduler
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)

# After each epoch:
# scheduler.step()
# print(f"LR: {scheduler.get_last_lr()}")

## 5. Gradient Clipping

In [ ]:
# Clip by global norm (common in RNNs/transformers)
def train_step_with_clipping(model, batch_x, batch_y, criterion, optimizer, max_norm=1.0):
  optimizer.zero_grad()
  logits = model(batch_x)
  loss = criterion(logits, batch_y)
  loss.backward()
  torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm)
  optimizer.step()
  return loss.item()

## 6. Saving & Loading

In [ ]:
# Save
torch.save({
  'model_state_dict': model.state_dict(),
  'optimizer_state_dict': optimizer.state_dict(),
  'epoch': epochs,
}, 'checkpoint.pt')

# Load
checkpoint = torch.load('checkpoint.pt', weights_only=True)
model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
print(f"Loaded from epoch {checkpoint['epoch']}")

## 7. Interview Checklist

- [ ] Always call `optimizer.zero_grad()` before `backward()`
- [ ] Use `model.train()` / `model.eval()` to toggle dropout & batch norm
- [ ] Wrap eval in `torch.no_grad()` or `@torch.inference_mode()`
- [ ] `CrossEntropyLoss` expects raw logits (no softmax)
- [ ] `clip_grad_norm_` for exploding gradients
- [ ] Save `state_dict()`, not the whole model